In [1]:
import os, re, sqlite3
import pandas as pd
from typing import Literal
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

load_dotenv()
MODEL = os.getenv("GEMINI_MODEL", "gemini-3.8-flash")
llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0)

In [2]:
#Prints all the supported models for use
from google import genai
client = genai.Client()
for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

response = llm.invoke("Explain what a vector embedding is in two sentences.")
print(response.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text', 'text': "A vector embedding is a numerical representation of complex data, such as text, images, or audio, translated into a list of numbers. This allows computers to understand the data's semantic meaning by placing conceptually similar items closer together in a multi-dimensional mathematical space.", 'extras': {'signature': 'Eo0RCooRAWkUfRNu7R26OzPEfHMk/e76dWZoX9fZPbkhfYrSP0ygecbw1V21xSjlXGEdfnQo8giunjeb1h8P8aFMPyFbOS+8HUvuQQIcMQjFu9d9C9wmDZWYcFKQNq0oLpF5IZ21D0NppetRturXOuG7u7HeSN+hYcItSHrWfb5aIIPNHtywx6YaYOYUaivYgaFpV7H2fZd5kA7qFjDRDAOq2V3r3Nou8+QtVSed5vX2BjJQ6lKWPs4rXVDKJzM4OiM2nhnVKsghh/UML4xy5QM5gfFXMAvh4LU6Zc/T5rOIvkQp57uvZJ3wau+UHKEtsrJ/ygNVG3gag0rtN7O1hMMnFMH3egEO9sHzFm55MJbgpL6X3Uv/ulPpVnfdhUNPwHt2w01TRhxYiXyNtP4zHoELQ3mAIBtiZ6iJCVp/URsEWLyj5VIuYAMhEbS6EPmvHKFiAa07FpQtzpIDtKW+aM7ojIXO3rnL0sg5l5DEIfbUN6SBkDk3vgayvsZ2SV+zb48i3+gQmqK1rPfUFB4mIJjxOy4RJ7hC6Rg9NafdtTzedNBoDc/s8yqh9/C5KtFSaVcD3sQgWr7qbDpxFxpeSSHUQy0o9EaOXG72J3v5xx3i4zONPB78SU8UrkL+iIqVijvQeuQTe/tN